# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Decision unit (my lane — Lane 2, refresh/content opportunity scoring): one row = one
content item for one client at a decision date**, described by what was measurable
*before* that date. The warehouse slice underneath is finer: `fact_content_daily_performance`
has **one row per `report_date` × `client_hash_id` × `content_hash_id`** (proved in §3), and I
aggregate it up to the decision unit.

**Tables:** `fact_content_daily_performance` (daily panel, 2025-01-27 → 2026-06-30,
partitioned by month), `dim_clients` (per-client history start dates — needed because
histories don't start together), `dim_content` (static item context).

**Time windows:** for the capstone, features come from a trailing window *before* the
decision date and the label from a window *after* it. In this contract I rehearse the
mechanics entirely inside one mid-panel month (`month=2026-03`, as the card asks):
**features = Mar 1–28**, **label window = Mar 29–31**. The June-2026 `_sample` table is the
panel's final month — the natural outcome window — so it is never touched here.

In [1]:
# Setup: repo, deps, DuckDB over the gated warehouse (token via cache/Colab secret — never in a cell).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "scikit-learn"], check=True)
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # Colab secrets panel, key HF_TOKEN
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # run from repo root locally

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"

print(con.sql(f"""
    SELECT COUNT(*) AS march_rows,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items
    FROM {MARCH}
""").df().to_string(index=False))
print()
print(con.sql(f"""
    SELECT (SELECT COUNT(*) FROM read_parquet('{BASE}/dim_clients.parquet'))  AS dim_clients_rows,
           (SELECT COUNT(*) FROM read_parquet('{BASE}/dim_content.parquet')) AS dim_content_rows
""").df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 march_rows  first_day   last_day  clients  content_items
    9841378 2026-03-01 2026-03-31       55         331437



 dim_clients_rows  dim_content_rows
              104            519606


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** (knowable before the decision date) | past-window aggregates of `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` (→ weighted avg position), active-day counts; static `dim_content.word_count`, `content_created_date` (→ age) | all computed from days ≤ the decision date |
| **Label / proxy** | `gsc_clicks` summed over the *future* window (Mar 29–31 here; ~30d after the decision date in the capstone) | the thing being predicted — never a feature, and nothing derived from it is either |
| **Context** | `client_hash_id`, `content_hash_id` (pseudonyms: group/join/split only), `report_date`, `month`, `gsc_data_available` / `ga4_data_available` (filters), `dim_clients.gsc_data_start` / `ga4_data_start` (window checks) | the model never learns from these |
| **Excluded** | `last_optimized_date`, `optimization_eligible_date` — FlyRank product-decision fields; training on them learns the product's own rule, not search reality. `sessions_ai` + `ai_*` — too sparse for label logic (lane guide warning); revisit as EDA only. `gsc_avg_position` daily column — `avg_position = 0` means "no data", not rank 0; I recompute a weighted position from sums instead. GA4 columns before a client's `ga4_data_start` — zero-FILLED, not zero. | each line above is the why |

In [2]:
# The guard that keeps label sources out of the features — same discipline as notebook w02.
FEATURES = ["impressions_28d", "clicks_28d", "ctr_28d", "wavg_position_28d", "active_days_28d"]
LABEL_SOURCES = {"future_clicks", "label_future_click"}
EXCLUDED = {"last_optimized_date", "optimization_eligible_date", "sessions_ai", "gsc_avg_position"}

assert not set(FEATURES) & LABEL_SOURCES, "leak: a label source is listed as a feature"
assert not set(FEATURES) & EXCLUDED, "an excluded field is listed as a feature"
print("feature set:", FEATURES)
print("label sources kept out:", sorted(LABEL_SOURCES))
print("excluded on purpose:", sorted(EXCLUDED))

feature set: ['impressions_28d', 'clicks_28d', 'ctr_28d', 'wavg_position_28d', 'active_days_28d']
label sources kept out: ['future_clicks', 'label_future_click']
excluded on purpose: ['gsc_avg_position', 'last_optimized_date', 'optimization_eligible_date', 'sessions_ai']


## 3. Verify it with queries (grain, counts, missing values, windows)

Three claims, three queries, outputs visible below:

1. **Grain** — one row really is one `report_date × client × content`: the duplicate probe
   returns **zero rows**.
2. **Size + span** — the March slice has **9,841,378 rows** spanning **Mar 1 → Mar 31 2026**
   (matches §1's output).
3. **Availability** — filtering with `IS TRUE` is not optional: only **36.7%** of March rows
   have GSC data available and just **4.2%** have GA4. The zeros outside those flags are
   *fill*, not measurements — every aggregate in this project carries the availability filter.

In [3]:
# Query 1 — grain probe: zero rows back = the stated grain holds.
dup = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY ALL HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Query 1 (grain probe): {len(dup)} duplicate keys found -> grain holds" if dup.empty
      else dup.to_string(index=False))

# Query 2 — row count + date span of my slice.
print("\nQuery 2 (count + span):")
print(con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS d0, MAX(report_date) AS d1 FROM {MARCH}")
      .df().to_string(index=False))

# Query 3 — availability, filtered with IS TRUE (zeros outside the flag are fill, not data).
print("\nQuery 3 (availability):")
print(con.sql(f"""
    SELECT COUNT(*)                                                    AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)          AS gsc_true,
           ROUND(AVG(CASE WHEN gsc_data_available IS TRUE THEN 1.0 ELSE 0 END), 3) AS gsc_share,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)          AS ga4_true,
           ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END), 3) AS ga4_share
    FROM {MARCH}
""").df().to_string(index=False))

Query 1 (grain probe): 0 duplicate keys found -> grain holds

Query 2 (count + span):


   rows         d0         d1
9841378 2026-03-01 2026-03-31

Query 3 (availability):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 all_rows  gsc_true  gsc_share  ga4_true  ga4_share
  9841378   3611061      0.367    413966      0.042


## 4. Five features, the quick score, and the deliberate leak

**Decision date: 2026-03-29.** Features aggregate Mar 1–28 per (client, content), GSC rows
only (`gsc_data_available IS TRUE`); the label is *"does the page get any search click in
Mar 29–31?"* — a future-window fact the features must not see.

The five features, each with its "knowable at the decision moment because…" line:

1. `impressions_28d` — sums impressions over days ≤ Mar 28.
2. `clicks_28d` — same window, clicks.
3. `ctr_28d` — ratio of the two above; no future term.
4. `wavg_position_28d` — `SUM(gsc_sum_position)/SUM(gsc_impressions)` over Mar 1–28
   (impression-weighted, so the `avg_position = 0` "no data" trap never enters).
5. `active_days_28d` — count of days ≤ Mar 28 with any impression.

**The trap, performed on purpose:** add `future_clicks` (the label's own source) as a sixth
"feature" → the score jumps to ~perfect; delete it → the honest number stays. That gap is
what leakage looks like on real warehouse data.

In [4]:
# One full-March scan, cached locally so reruns don't re-scan (rate-limit rule).
CACHE = "work/outputs/w03_march_feature_frame.parquet"
os.makedirs("work/outputs", exist_ok=True)
if os.path.exists(CACHE):
    frame = con.sql(f"SELECT * FROM read_parquet('{CACHE}')").df()
    print(f"loaded cache: {CACHE}")
else:
    frame = con.sql(f"""
        WITH past AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions)                                   AS impressions_28d,
                   SUM(gsc_clicks)                                        AS clicks_28d,
                   SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS ctr_28d,
                   SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS wavg_position_28d,
                   COUNT(*) FILTER (WHERE gsc_impressions > 0)            AS active_days_28d
            FROM {MARCH}
            WHERE report_date <= DATE '2026-03-28' AND gsc_data_available IS TRUE
            GROUP BY 1, 2
            HAVING SUM(gsc_impressions) >= 10
        ),
        fut AS (
            SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS future_clicks
            FROM {MARCH}
            WHERE report_date > DATE '2026-03-28' AND gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT past.*, COALESCE(fut.future_clicks, 0) AS future_clicks
        FROM past LEFT JOIN fut USING (client_hash_id, content_hash_id)
    """).df()
    con.execute(f"COPY (SELECT * FROM frame) TO '{CACHE}' (FORMAT PARQUET)")
    print(f"scanned March once, cached to {CACHE}")

frame["ctr_28d"] = frame["ctr_28d"].fillna(0)
frame["wavg_position_28d"] = frame["wavg_position_28d"].fillna(frame["wavg_position_28d"].median())
frame["label_future_click"] = (frame["future_clicks"] > 0).astype(int)
print(f"{len(frame):,} (client, content) rows | positives: {frame['label_future_click'].mean():.3f}")
print(frame[FEATURES].describe().round(2).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

scanned March once, cached to work/outputs/w03_march_feature_frame.parquet
139,757 (client, content) rows | positives: 0.186
       impressions_28d  clicks_28d    ctr_28d  wavg_position_28d  active_days_28d
count        139757.00   139757.00  139757.00          139757.00        139757.00
mean           1779.98        5.27       0.00              16.64            22.57
std            5321.89       26.69       0.01              17.46             7.27
min              10.00        0.00       0.00               0.00             1.00
25%              74.00        0.00       0.00               5.18            18.00
50%             320.00        0.00       0.00               8.97            26.00
75%            1390.00        3.00       0.00              22.05            28.00
max          507809.00     4956.00       0.40             175.10            28.00


In [5]:
# Quick score (grouped by client, so no client straddles the split) — then the trap.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
              .split(frame, groups=frame["client_hash_id"]))
train, test = frame.iloc[tr], frame.iloc[te]
y_tr, y_te = train["label_future_click"], test["label_future_click"]

def quick_auc(cols):
    m = RandomForestClassifier(n_estimators=60, max_depth=8, n_jobs=-1, random_state=42)
    m.fit(train[cols], y_tr)
    return roc_auc_score(y_te, m.predict_proba(test[cols])[:, 1])

honest = quick_auc(FEATURES)
leaky  = quick_auc(FEATURES + ["future_clicks"])   # the deliberate leak: the label's own source
print(f"honest AUC (5 past-window features, client-grouped split): {honest:.3f}")
print(f"leaky  AUC (+ future_clicks, the label source)           : {leaky:.3f}")
print(f"\nThe jump of {leaky - honest:+.3f} toward perfect is the leak. future_clicks is now deleted")
print("from the feature list; the honest number is the one that stands:", f"{honest:.3f}")
assert "future_clicks" not in FEATURES

honest AUC (5 past-window features, client-grouped split): 0.892
leaky  AUC (+ future_clicks, the label source)           : 1.000

The jump of +0.108 toward perfect is the leak. future_clicks is now deleted
from the feature list; the honest number is the one that stands: 0.892


## 4b. Data limits

What this data can never tell me, and the two limits I have to design around:

- **Histories don't start together.** `gsc_data_start` varies by client (queried below) — a
  fixed global calendar window silently gives some clients months of implicit zeros. The
  capstone must use per-client windows checked against `dim_clients`.
- **GA4 columns are mostly fill, not data** — `ga4_data_available IS FALSE` on **95.8%** of
  March rows (measured below). Zeros there mean *not measured*, not "no engagement";
  GA4-based features are usable only on the small measured slice.
- **This month-slice is a rehearsal, not the design.** A 3-day label window inside one month
  is only for mechanics; the real label needs ~30 future days across month partitions, with
  the June-2026 `_sample`/final month sealed as test.
- **Hashed IDs mean no *why*.** The warehouse says what moved, never why — no page text, no
  queries in the clear. All conclusions stay observational and decision-support.

In [6]:
# The named limitation, quantified: client history depth is wildly uneven.
print(con.sql(f"""
    SELECT MIN(gsc_data_start)  AS earliest_gsc_start,
           MAX(gsc_data_start)  AS latest_gsc_start,
           COUNT(*) FILTER (WHERE gsc_data_start > DATE '2025-06-30') AS clients_starting_after_2025h1,
           COUNT(*)             AS clients_total
    FROM read_parquet('{BASE}/dim_clients.parquet')
    WHERE has_gsc_access IS TRUE
""").df().to_string(index=False))
print()
print(con.sql(f"""
    SELECT ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 0.0 ELSE 1.0 END), 3)
           AS march_rows_with_ga4_fill_not_data
    FROM {MARCH}
""").df().to_string(index=False))

earliest_gsc_start latest_gsc_start  clients_starting_after_2025h1  clients_total
        2025-01-27       2026-06-02                             51             67



 march_rows_with_ga4_fill_not_data
                             0.958


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.